# Machine Health — solución del equipo Acosta · Borges · PelinskiRecorrido completo: auditoría de la métrica, EDA, ingeniería de features física,validación bajo cambio de régimen y generación de la entrega.El resultado central de este notebook no es el modelo sino el **esquema devalidación**. Una primera entrega, validada con `GroupKFold` por `machine_id`,estimaba 47 puntos y obtuvo 28 en el servidor, por debajo del baseline trivialde 29. La sección 4 identifica la causa y la corrige.

## 0. Entorno y carga

In [ ]:
import sys, warnings, numpy as np, pandas as pdwarnings.filterwarnings("ignore")import lightgbm as lgbfrom sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifierfrom sklearn.linear_model import LogisticRegressionfrom sklearn.pipeline import make_pipelinefrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import StratifiedGroupKFoldDATA = "datos_sinraw/"       # ajustarKIT  = "participant_kit/"    # ajustarsys.path.insert(0, KIT)from scoring import (compute_score, validate_prediction_package,                     FAULT_IDS, LABEL_COLUMNS, FAMILIES, COSTS)RS = 42np.random.seed(RS)train = pd.read_parquet(DATA + "train_sup.parquet")test  = pd.read_parquet(DATA + "test.parquet")FA = FAULT_IDSprint("train", train.shape, "| test", test.shape)

## 1. EDA estructuralSe verifica, no se asume. Cuatro hechos condicionan todo el diseño.

In [ ]:
lab = train[LABEL_COLUMNS].to_numpy()n_fallas = lab.sum(1)estado = np.where(n_fallas == 0, 0, lab.argmax(1) + 1)   # 0 = sano, k = F{k}ESTADOS = ["sano"] + FAprint("1. nulos en features:", int(train.isna().sum().sum()), "| duplicados:", int(train.duplicated().sum()))print("2. is_combo:", train.is_combo.sum(), "-> fallas por ventana:",      pd.Series(n_fallas).value_counts().sort_index().to_dict())print("3. máquinas train/test:", train.machine_id.nunique(), test.machine_id.nunique(),      "| intersección:", len(set(train.machine_id) & set(test.machine_id)))nun = train.groupby("session_id")["window_id"].size()est_por_ses = train.assign(e=estado).groupby("session_id").e.nunique()print("4. ventanas por sesión:", nun.unique(), "| sesiones con un solo estado:",      int((est_por_ses == 1).sum()), "de", len(est_por_ses))

Cuatro consecuencias:1. **Datos limpios.** No hay imputación que hacer; las columnas `*__nan_frac` son   features de calidad de adquisición, no metadatos.2. **`is_combo = 0` en las 1750 filas.** El problema declarado como multilabel es,   en el train, **multiclase de 14 estados mutuamente excluyentes**. Se modela con   softmax: impone `Σp ≤ 1`, que es lo que consume la regla de decisión.3. **Las máquinas de train y test son disjuntas.** Validar agrupando por máquina es   obligatorio.4. **La etiqueta es constante dentro de cada sesión (250 de 250), y cada sesión tiene   7 ventanas.** El tamaño efectivo de la muestra es **250 sesiones, no 1750 ventanas**.   El riesgo de sobreajuste es mucho mayor de lo que sugiere el tamaño nominal.

### 1.1 Prevalencias y severidad

In [ ]:
prev = train[LABEL_COLUMNS].mean().rename(lambda s: s.replace("label_", ""))sev = pd.concat([train.loc[train[f"label_{f}"] == 1, f"severity_{f}"] for f in FA])print(prev.round(4).to_string())print("\nsuma de prevalencias:", round(prev.sum(), 3),      "| ventanas sanas:", round((n_fallas == 0).mean(), 3))print("de las ventanas con falla, severas (sev>0.5):", round((sev > 0.5).mean(), 3))

### 1.2 Cambio de régimen entre train y testLas máquinas de test no sólo son otras: **operan en otro punto de trabajo**.

In [ ]:
for c in ["rpm_mean", "flow_mean", "Tamb", "temp_winding__mean", "delta_p_mean"]:    print("%-22s train %8.1f   test %8.1f" % (c, train[c].mean(), test[c].mean()))

`rpm_mean` pasa de 1742 a 2052 y la temperatura ambiente sube 3 °C. Las magnitudesabsolutas de los sensores **no transfieren**: la energía de vibración escala con elcuadrado de la velocidad, así que una máquina rápida y sana se parece, en valorabsoluto, a una máquina lenta en falla. Esto ordena la sección 3.

## 2. Auditoría de la métrica`final = 0,70·cost + 0,20·prob + 0,10·diag`. El 70 % lo decide una regla fija yconocida que transforma las probabilidades entregadas en una acción de mantenimiento.Con `S = Σ p_f` y `S_g` la masa de la familia `g`:| Acción | Costo esperado que evalúa la regla ||---|---|| MONITOR | `10·S` || DERATE | `7 + 3,5·S` || STOP | `12` || INSPECT(g) | `4 + 8·(S − S_g)` |Con un softmax, `S ≤ 1`, de modo que **DERATE y STOP nunca se eligen**: el juego reales MONITOR contra INSPECT-familia.

In [ ]:
FAMS = sorted(set(FAMILIES.values()))FIDX = {g: np.array([i for i, f in enumerate(FA) if FAMILIES[f] == g]) for g in FAMS}def acciones(P13):    S = P13.sum(1); n = len(P13)    c = np.empty((n, 3 + len(FAMS)))    c[:, 0] = 10 * S; c[:, 1] = 7 + 3.5 * S; c[:, 2] = 12    for k, g in enumerate(FAMS):        c[:, 3 + k] = 4 + 8 * (S - P13[:, FIDX[g]].sum(1))    nom = np.array(["MONITOR", "DERATE", "STOP"] + ["INSPECT_" + g for g in FAMS])    return pd.Series(nom[c.argmin(1)])P0 = np.tile(prev.to_numpy(), (len(train), 1))print("acciones que elige la regla para el baseline P0:")print(acciones(P0).value_counts().to_string())

**Corrección al material del kit.** `baseline.ipynb` afirma que "la acción óptima paraprobabilidades bajas es MONITOR". Es falso para su propio baseline: con lasprevalencias, `S = 0,876` y `S_mecánica = 0,464`, con lo que INSPECT mecánica cuesta`4 + 8·0,412 = 7,30` contra `8,76` de MONITOR. La regla elige **INSPECT mecánica enlas 1848 ventanas**.Esto importa porque fija la vara real: el baseline no es pasivo, es un modelo quesiempre inspecciona la familia más frecuente. Para superarlo hay que acertar lafamilia **más del 46 % de las veces**, que es la prevalencia de la familia mecánica.

### 2.1 El costo depende de la FAMILIA, no de la falla exactaCon una sola familia verdadera presente, el costo real es:| Estado real | MONITOR | INSPECT correcta | INSPECT incorrecta ||---|---|---|---|| Sana | 0 | 4 | 4 || Falla leve | 5 | 4 | 12 || Falla severa | 15 | 4 | 12 |Distinguir F03 de F04 no aporta un centavo al `cost_score`; sólo al `diag_score`, quepesa 10 %. **El 70 % del puntaje se juega en acertar una de cinco familias**(sana, mecánica, estructural, eléctrica, hidráulica). Por eso todo el análisisposterior reporta la exactitud de familia junto al score.

### 2.2 Piso y techo

In [ ]:
def evaluar(P13, df=train, idx=None):    idx = np.arange(len(df)) if idx is None else idx    e = pd.DataFrame({"window_id": df.window_id.iloc[idx].to_numpy()})    for j, f in enumerate(FA): e[f] = np.clip(P13[idx][:, j], 0, 1)    return compute_score(df.iloc[idx], e)["overall"]r0 = evaluar(P0); ro = evaluar(train[LABEL_COLUMNS].to_numpy(float))print("P0 prevalencia  final=%6.2f  cost=%6.2f" % (r0["final_score"], r0["cost_score"]))print("oráculo         final=%6.2f  cost=%6.2f" % (ro["final_score"], ro["cost_score"]))print("\nnaive_cost=%.2f. Aun acertando todo hay que pagar la inspección de $4," % r0["naive_cost"])print("por eso cost_score se planta en %.1f y el final máximo del problema es %.1f."      % (ro["cost_score"], ro["final_score"]))

El máximo alcanzable es ≈74, no 100. Comparar contra 100 es engañoso; se reporta elporcentaje del techo real.

## 3. Ingeniería de features físicaCada bloque apunta a un modo de falla y está construido para ser **adimensional orelativo**, que es la condición para transferir a máquinas y regímenes distintos.La justificación física de cada grupo:- **Vibración normalizada por régimen.** La energía vibratoria escala con el cuadrado  de la velocidad, de ahí `rms/(rpm/1000)²`. Sin esto el modelo confunde "máquina  rápida" con "máquina en falla", que es precisamente el shift medido en 1.2.  La fracción de energía en `1x` marca **desbalance (F01)**; la relación `2x/1x`,  **desalineación (F02)**, porque la desalineación excita el segundo armónico.  `log1p(kurtosis)` y `crest` capturan los impactos de los **rodamientos (F03–F05)`.  La relación axial/radial separa F02 de F01, y la asimetría entre apoyos marca  **pérdida de rigidez (F07)**.- **Eléctricas (F08–F11).** Desbalance de tensión y corriente entre las tres fases  como `(max−min)/media` para **desequilibrio (F08)**; `I_min/I_media` para  **pérdida de fase (F09)**; corriente relativa a la velocidad (carga) para  **barras rotóricas (F10)**.- **Térmicas.** Siempre como diferencia contra `Tamb`, nunca en valor absoluto: el  ambiente del test es 3 °C más cálido. La sobretemperatura de devanado apunta a  **cortocircuito entre espiras (F11)**; la diferencia entre rodamientos, a  **lubricación deficiente (F06)**.- **Hidráulicas (F12–F13).** Leyes de afinidad de bombas: altura adimensional  `Δp/(rpm/1000)²` y coeficiente de caudal `flow/rpm` ubican el punto de operación en  la curva normalizada. El rendimiento aparente `flow·Δp/(V·I)` marca  **obstrucción o fuga (F13)**; la presión de succión deprimida, **cavitación (F12)**.

In [ ]:
BASE = [c for c in test.columns        if c not in ("window_id", "machine_id", "session_id", "timestamp_start_s")]ACC = ["acc_radial_a", "acc_radial_b", "acc_axial"]def fisicas(df):    X = pd.DataFrame(index=df.index)    rpm = df.rpm_mean.clip(lower=1) / 1000.0    rpm2 = rpm ** 2    for a in ACC:        rms = df[f"{a}__rms"].clip(lower=1e-9)        X[f"{a}__rms_n"] = rms / rpm2        X[f"{a}__1x_r"]  = df[f"{a}__1x"] / (rms ** 2 + 1e-9)        X[f"{a}__2x1x"]  = df[f"{a}__2x"] / (df[f"{a}__1x"].abs() + 1e-6)        X[f"{a}__lkurt"] = np.log1p(df[f"{a}__kurtosis"].clip(lower=0))        X[f"{a}__crest_n"] = df[f"{a}__crest"]    ra = df["acc_radial_a__rms"].clip(lower=1e-9)    X["ax_rad"]  = df["acc_axial__rms"] / ra    X["rad_b_a"] = df["acc_radial_b__rms"] / ra    X["vib_tot"] = df[[f"{a}__rms" for a in ACC]].sum(1) / rpm2    V = df[["voltage_a__rms", "voltage_b__rms", "voltage_c__rms"]]    I = df[["current_a__rms", "current_b__rms", "current_c__rms"]]    vm, im = V.mean(1).clip(lower=1e-9), I.mean(1).clip(lower=1e-9)    X["V_desbal"] = (V.max(1) - V.min(1)) / vm    X["I_desbal"] = (I.max(1) - I.min(1)) / im    X["V_cv"] = V.std(1) / vm    X["I_cv"] = I.std(1) / im    X["I_min_r"] = I.min(1) / im    X["I_max_r"] = I.max(1) / im    X["S_ap"] = im * vm / 1000.0    X["I_por_rpm"]  = im / rpm    X["I_por_flow"] = im / (df.flow_mean.abs() + 1e-6)    X["dT_wind"]   = df["temp_winding__mean"] - df.Tamb    X["dT_b1"]     = df["temp_bearing1__mean"] - df.Tamb    X["dT_b2"]     = df["temp_bearing2__mean"] - df.Tamb    X["dT_b1b2"]   = df["temp_bearing1__mean"] - df["temp_bearing2__mean"]    X["dT_wind_b"] = df["temp_winding__mean"] - df[["temp_bearing1__mean", "temp_bearing2__mean"]].mean(1)    X["dT_wind_rpm"] = X["dT_wind"] / rpm    X["head_n"]    = df.delta_p_mean / rpm2    X["flow_n"]    = df.flow_mean / rpm    X["p_in_n"]    = df.pressure_in_mean / rpm2    X["p_ratio"]   = df.pressure_out_mean / (df.pressure_in_mean.abs() + 1e-6)    X["hidr_pot"]  = df.flow_mean * df.delta_p_mean / (im * vm + 1e-6)    X["flow_head"] = df.flow_mean / (df.delta_p_mean.abs() + 1e-6)    X["rpm_cv"] = df.rpm_std / df.rpm_mean.clip(lower=1)    nanc = [c for c in df.columns if c.endswith("__nan_frac")]    X["nan_tot"] = df[nanc].sum(1)    X["nan_max"] = df[nanc].max(1)    return X.replace([np.inf, -np.inf], np.nan).fillna(0.0)def rank_maq(df, cols):    r = df.groupby("machine_id")[cols].rank(pct=True)    r.columns = [c + "__r" for c in cols]    return rdef construir(df, modo="fis"):    if modo == "base":        return df[BASE]    todo = pd.concat([df[BASE], fisicas(df)], axis=1)    if modo == "fis":        return todo    return pd.concat([todo, rank_maq(pd.concat([df[["machine_id"]], todo], axis=1),                                     list(todo.columns))], axis=1)print("base:", len(BASE), "| físicas:", fisicas(train).shape[1],      "| total:", construir(train).shape[1])

## 4. El esquema de validaciónÉsta es la sección decisiva. Una primera entrega validada con `GroupKFold` por`machine_id` estimaba 47 puntos y sacó 28 en el servidor. La causa: `GroupKFold`mide **máquina nueva, mismo régimen**, mientras que el test es **máquina nueva,régimen nuevo** (1742 → 2052 rpm).Para medir lo que realmente importa se construye una segunda validación: se ordenanlas 26 máquinas por `rpm_mean`, se entrena con las 13 lentas y se valida con las 13rápidas, cuyo régimen medio (2057 rpm) coincide con el del test (2052).

In [ ]:
rpm_maq = train.groupby("machine_id").rpm_mean.mean().sort_values()rapidas = set(rpm_maq.index[13:])es_rap  = train.machine_id.isin(rapidas).to_numpy()IDX_SHIFT = (np.where(~es_rap)[0], np.where(es_rap)[0])print("rpm medio -> 13 lentas %.0f | 13 rápidas %.0f | test %.0f"      % (rpm_maq.iloc[:13].mean(), rpm_maq.iloc[13:].mean(), test.rpm_mean.mean()))FAM_DE = np.array(["sano"] + [FAMILIES[f] for f in FA])def acc_familia(P14, idx):    return float((FAM_DE[estado[idx]] == FAM_DE[P14[idx].argmax(1)]).mean())def mk_lgb():    return lgb.LGBMClassifier(objective="multiclass", num_class=14, n_estimators=500,        learning_rate=0.05, num_leaves=15, min_child_samples=25, subsample=0.9,        subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0, verbose=-1, random_state=RS)def oof_gkf(X, ctor, seeds=(42, 2024, 7)):    acc = np.zeros((len(train), 14))    for s in seeds:        cv = StratifiedGroupKFold(5, shuffle=True, random_state=s)        o = np.zeros((len(train), 14))        for a, b in cv.split(X, estado, train.machine_id.to_numpy()):            m = ctor(); m.fit(X.iloc[a], estado[a]); o[b] = m.predict_proba(X.iloc[b])        acc += o    return acc / len(seeds)def oof_shift(X, ctor):    a, b = IDX_SHIFT    o = np.zeros((len(train), 14))    m = ctor(); m.fit(X.iloc[a], estado[a]); o[b] = m.predict_proba(X.iloc[b])    return o

In [ ]:
filas = []for modo in ("base", "fis", "fis_rank"):    X = construir(train, modo)    g, s = oof_gkf(X, mk_lgb), oof_shift(X, mk_lgb)    rg = evaluar(g[:, 1:]); rs = evaluar(s[:, 1:], idx=IDX_SHIFT[1])    filas.append(dict(features=modo,                      GKF=rg["final_score"], GKF_fam=acc_familia(g, np.arange(len(train))),                      SHIFT=rs["final_score"], SHIFT_fam=acc_familia(s, IDX_SHIFT[1])))pd.DataFrame(filas).round(3)

| Features | GroupKFold | Shift de régimen ||---|---|---|| Crudas | 44,11 | 38,09 || **Físicas** | 49,82 | **45,35** || Físicas + rango por máquina | **51,88** | 41,80 |Dos lecturas, y la segunda es la que salva la entrega:1. **Las features físicas ganan en los dos esquemas** (+5,7 y +7,3). Es la palanca   real: normalizar por régimen es lo que permite que el conocimiento aprendido en   máquinas lentas se aplique a máquinas rápidas.2. **El rango percentil por máquina es una trampa de validación.** Es lo mejor bajo   `GroupKFold` (51,88, el número más alto de todo el notebook) y de lo peor bajo   cambio de régimen (41,80). Optimizar contra `GroupKFold` lleva a elegirlo, y es   exactamente el mecanismo por el cual la primera entrega estimó 47 y obtuvo 28.Se descarta la normalización por máquina. La decisión se toma **contra el esquema decambio de régimen**, no contra el que da el número más alto.

## 5. Comparación de modelos y ensambleRegla del equipo: ninguna diferencia se acepta si no supera el desvío entre folds.

In [ ]:
MODELOS = {    "lgb": mk_lgb,    "et":  lambda: ExtraTreesClassifier(n_estimators=600, min_samples_leaf=3,             max_features="sqrt", n_jobs=-1, random_state=RS),    "rf":  lambda: RandomForestClassifier(n_estimators=600, min_samples_leaf=3,             max_features="sqrt", n_jobs=-1, random_state=RS),    "lr":  lambda: make_pipeline(StandardScaler(),             LogisticRegression(max_iter=3000, C=0.3, random_state=RS)),}X = construir(train, "fis")OG, OS, filas = {}, {}, []for nom, ctor in MODELOS.items():    OG[nom], OS[nom] = oof_gkf(X, ctor), oof_shift(X, ctor)    filas.append(dict(modelo=nom,                      GKF=evaluar(OG[nom][:, 1:])["final_score"],                      SHIFT=evaluar(OS[nom][:, 1:], idx=IDX_SHIFT[1])["final_score"]))pd.DataFrame(filas).round(2)

In [ ]:
def mez(d, ks):    P = np.mean([d[k] for k in ks], 0); return P / P.sum(1, keepdims=True)COMBOS = {"lgb+et": ["lgb", "et"], "lgb+rf+lr": ["lgb", "rf", "lr"],          "lgb+et+rf+lr": ["lgb", "et", "rf", "lr"], "lgb+et+lr": ["lgb", "et", "lr"]}filas = [dict(ensamble=n,              GKF=evaluar(mez(OG, k)[:, 1:])["final_score"],              SHIFT=evaluar(mez(OS, k)[:, 1:], idx=IDX_SHIFT[1])["final_score"])         for n, k in COMBOS.items()]pd.DataFrame(filas).round(2)

`lgb+et` es el mejor ensamble bajo cambio de régimen (44,92) y queda a 0,4 deLightGBM solo (45,35), diferencia muy inferior al ruido. Se elige el ensamble porser la opción de menor varianza: promediar dos familias de árboles con sesgosdistintos protege ante la partición particular.

### 5.1 Dos ideas que se probaron y se descartanSe registran porque el descarte con evidencia también es resultado.

In [ ]:
PREV = np.r_[1 - n_fallas.mean(), prev.to_numpy()]def encoge(P, w):    Q = (1 - w) * P + w * PREV[None, :]; return Q / Q.sum(1, keepdims=True)PG, PS = mez(OG, ["lgb", "et"]), mez(OS, ["lgb", "et"])filas = [dict(w=w,              GKF=evaluar(encoge(PG, w)[:, 1:])["final_score"],              SHIFT=evaluar(encoge(PS, w)[:, 1:], idx=IDX_SHIFT[1])["final_score"])         for w in (0.0, 0.1, 0.2, 0.3, 0.45)]pd.DataFrame(filas).round(2)

**Encogimiento hacia la prevalencia.** La hipótesis era que, con la familia incierta,conviene no apostar (INSPECT equivocado cuesta 12 contra 5 de MONITOR). Empeoramonótonamente en los dos esquemas: se descarta.**Afilado de probabilidades contra la métrica.** Se probó elegir el exponente encuatro folds y aplicarlo al quinto: 47,48 contra 47,17 sin transformar, con desvíoentre folds de 1,7. Está dentro del ruido. La regla de decisión ya opera sobreprobabilidades bien calibradas; distorsionarlas no paga.

### 5.2 Promedio por sesiónLa etiqueta es constante en las 7 ventanas de cada sesión (verificado en 1). Promediarlas probabilidades de la sesión reduce varianza sin usar información prohibida: sólousa `session_id`, que viene en el test.

In [ ]:
def por_sesion(P, df):    L = pd.DataFrame(np.log(np.clip(P, 1e-9, 1))); L["s"] = df.session_id.to_numpy()    M = np.exp(L.groupby("s").transform("mean").to_numpy())    return M / M.sum(1, keepdims=True)print("sin sesión  GKF=%.2f  SHIFT=%.2f"      % (evaluar(PG[:, 1:])["final_score"], evaluar(PS[:, 1:], idx=IDX_SHIFT[1])["final_score"]))print("con sesión  GKF=%.2f  SHIFT=%.2f"      % (evaluar(por_sesion(PG, train)[:, 1:])["final_score"],         evaluar(por_sesion(PS, train)[:, 1:], idx=IDX_SHIFT[1])["final_score"]))

El efecto es neutro en magnitud (+0,24 y −0,16), pero se conserva por el argumentoestructural: la etiqueta es constante por sesión, de modo que el promedio eliminaruido de ventana sin introducir sesgo.

## 6. EntregaSe reentrena con todo `train_sup` y se predice el test. Ninguna etiqueta del testinterviene en ningún paso, y ninguna transformación usa estadísticos calculadossobre el test.

In [ ]:
Xtr, Xte = construir(train, "fis"), construir(test, "fis")assert list(Xtr.columns) == list(Xte.columns), "columnas desalineadas"P = np.zeros((len(test), 14))for nom in ["lgb", "et"]:    m = MODELOS[nom](); m.fit(Xtr, estado); P += m.predict_proba(Xte)P /= 2P = por_sesion(P, test)sub = pd.DataFrame({"window_id": test.window_id.to_numpy()})for j, f in enumerate(FA):    sub[f] = np.clip(P[:, j + 1], 0, 1)sub = validate_prediction_package(test, sub)sub.to_csv("submit.csv", index=False)print("submit.csv", sub.shape, "validado")print("suma de probabilidades por fila: %.3f | prevalencia del train: %.3f"      % (sub[FA].sum(1).mean(), prev.sum()))print("\nacciones que elegirá la regla sobre el test:")print(acciones(sub[FA].to_numpy()).value_counts().to_string())

## 7. Resumen de decisiones| Decisión | Evidencia ||---|---|| Multiclase de 14 estados (softmax) | `is_combo = 0` en las 1750 filas || Validación por cambio de régimen, no sólo `GroupKFold` | `GroupKFold` estimó 47 y el servidor dio 28 || Features físicas adimensionales | +5,7 en `GroupKFold` y +7,3 bajo cambio de régimen || Se descarta el rango percentil por máquina | mejor en `GroupKFold` (51,9) y peor bajo cambio de régimen (41,8) || Ensamble `lgb+et` | menor varianza, empata al mejor individual || Promedio por sesión | la etiqueta es constante en las 7 ventanas || Se descarta el encogimiento hacia la prevalencia | empeora en los dos esquemas || Se descarta el afilado de probabilidades | +0,31 contra un desvío entre folds de 1,7 || Se descarta el tuneo de hiperparámetros | el cambio de familia de modelo mueve menos que el ruido de partición |El tamaño efectivo de la muestra es de 250 sesiones. Con esa cifra, cualquier ajustede muchos parámetros contra la métrica es sobreajuste, y por eso las mejoras aceptadasson estructurales (features con sentido físico, esquema de validación) y no numéricas.